# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-19 11:27:46 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-19


In [2]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-19 11:27:51 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
get_market_signals() function de

In [3]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 232331 rows (product x cohort)
  With market data: 44435
  With WAC: 76010
  With target margin: 232331


In [4]:
lookup[(lookup['current_price'] > lookup['market_max'])&(lookup['total_stocks']>0) ].to_excel('manual_data.xlsx')

In [8]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(2829,703,'market_25'),
(6802,704,'market_75'),
(12870,700,'market_25'),
(12419,701,'market_75'),
(5243,1125,'market_25'),
(22038,1125,'market_25'),
(13444,1123,'market_max'),
(6098,1123,'market_50'),
(6098,1124,'market_50'),
(6098,1125,'market_50'),
(6098,1126,'market_50'),
(10397,700,'market_25'),
(10397,701,'market_25'),
(10397,704,'market_25'),
(10397,1126,'market_25'),
(12338,702,'market_25'),
(2193,700,'market_25'),
(5736,700,'market_50'),
(8356,1125,'market_50'),
(13024,1123,'market_25'),
(6561,700,'market_25'),
(3028,700,'market_25'),
(3028,701,'market_25'),
(3028,702,'market_25'),
(3028,703,'market_25'),
(3028,704,'market_25'),
(3028,1125,'market_25'),
(3028,1126,'market_25'),
(14032,702,'market_50'),
(14035,702,'market_50'),
(7036,700,'market_25'),
(7036,701,'market_25'),
(7036,702,'market_25'),
(7036,703,'market_25'),
(7036,704,'market_25'),
(7036,1123,'market_25'),
(1744,1124,'market_25'),
(1064,700,'market_75'),
(1064,701,'market_75'),
(1064,1124,'market_75'),
(1064,1125,'market_75'),
(8155,701,'market_25'),
(6097,704,'market_max'),
(11936,704,'market_50'),
(11974,1123,'market_max'),
(11974,1124,'market_max'),
(11974,1125,'market_max'),
(11974,1126,'market_max'),
(6200,700,'market_25'),
(12647,700,'market_25'),
(5391,700,'market_25'),
(6803,703,'market_75'),
(6266,704,'market_50'),
(22878,700,'market_25'),
(22878,1126,'market_25'),
(11931,700,'market_25'),
(13326,700,'market_25'),
(13326,701,'market_25'),
(13326,703,'market_25'),
(13326,704,'market_25'),
(22867,704,'market_25'),
(10938,700,'market_50'),
(19932,700,'market_25'),
(19932,701,'market_25'),
(19932,1123,'market_25'),
(11183,700,'market_50'),
(12536,701,'market_25'),
(12664,1123,'market_50'),
(12664,1124,'market_50'),
(12664,1125,'market_50'),
(12664,1126,'market_50'),
(12665,1123,'market_50'),
(12665,1125,'market_50'),
(12665,1126,'market_50'),
(10722,700,'market_25'),
(10722,701,'market_25'),
(10722,702,'market_25'),
(10722,703,'market_25'),
(10722,704,'market_25'),
(10722,1123,'market_25'),
(10722,1125,'market_25'),
(10722,1126,'market_25'),
(13452,1123,'market_50'),
(13454,1123,'market_max'),
(7007,704,'market_25'),
(7007,1125,'market_25'),
(13025,701,'market_25'),
(6422,700,'market_25'),
(13325,700,'market_25'),
(13325,701,'market_25'),
(13325,703,'market_25'),
(13325,704,'market_25'),
(13325,1125,'market_25'),
(8638,700,'market_25'),
(8638,701,'market_25'),
(8638,703,'market_25'),
(8638,1123,'market_25'),
(8638,1124,'market_25'),
(8638,1125,'market_25'),
(8638,1126,'market_25'),
(2663,704,'market_25'),
(523,701,'market_25'),
(9375,700,'market_25'),
(9375,701,'market_25'),
(9375,702,'market_25'),
(9375,703,'market_25'),
(9375,704,'market_25'),
(9375,1126,'market_25'),
(11160,700,'market_50'),
(10721,1125,'market_75'),
(9379,704,'market_25'),
(4032,701,'market_25'),
(13440,700,'market_max'),
(13442,1123,'market_50'),
(13443,1123,'market_max'),
(6343,700,'market_25'),
(7887,1124,'market_25'),
(6195,701,'market_50'),
(4327,701,'market_50'),
(13853,700,'market_25'),
(13853,1125,'market_25'),
(1514,700,'market_25'),
(1514,701,'market_25'),
(1514,702,'market_25'),
(1514,703,'market_25'),
(1514,1123,'market_25'),
(1514,1125,'market_25'),
(1514,1126,'market_25'),
(5644,702,'market_25'),
(14033,702,'market_50'),
(12489,701,'market_50'),
(2505,700,'market_75'),
(12676,703,'market_50'),
(12494,700,'market_25'),
(12494,701,'market_25'),
(12494,702,'market_25'),
(12494,703,'market_25'),
(12494,704,'market_25'),
(12494,1123,'market_25'),
(12494,1124,'market_25'),
(12494,1125,'market_25'),
(12494,1126,'market_25'),
(6728,700,'market_25'),
(6728,701,'market_25'),
(6728,702,'market_25'),
(6728,703,'market_25'),
(6728,704,'market_25'),
(6728,1123,'market_25'),
(6728,1124,'market_25'),
(6728,1125,'market_25'),
(962,1126,'market_50'),
(10181,703,'market_25'),
(10181,704,'market_25'),
(13113,700,'market_25'),
(13113,701,'market_25'),
(13113,702,'market_25'),
(13113,1123,'market_25'),
(13113,1126,'market_25'),
(5020,700,'market_75'),
(10582,702,'market_25'),
(10582,703,'market_25'),
(10582,1123,'market_25'),
(10582,1124,'market_25'),
(10582,1125,'market_25'),
(10582,1126,'market_25'),
(3395,700,'market_25'),
(9295,700,'market_25'),
(9126,701,'market_25'),
(11180,700,'market_25'),
(11771,700,'market_25'),
(11771,703,'market_25'),
(11771,704,'market_25'),
(7526,701,'market_50'),
(9816,700,'market_50'),
(9816,701,'market_50'),
(9816,702,'market_75'),
(9816,703,'market_75'),
(9816,704,'market_75'),
(9816,1123,'market_50'),
(9816,1124,'market_50'),
(9816,1125,'market_50'),
(9816,1126,'market_50'),
(9815,700,'market_50'),
(9815,701,'market_50'),
(9815,702,'market_75'),
(9815,703,'market_75'),
(9815,704,'market_75'),
(9815,1123,'market_50'),
(9815,1124,'market_50'),
(9815,1125,'market_50'),
(9815,1126,'market_50'),
(23032,702,'market_25'),
(23032,1125,'market_25'),
(5831,702,'market_25'),
(2731,700,'market_25'),
(12232,702,'market_max'),
(6257,701,'market_50'),
(5128,702,'market_25'),
(202,700,'market_75'),
(1722,700,'market_25'),
(1722,701,'market_25'),
(1722,702,'market_25'),
(1722,703,'market_25'),
(1722,704,'market_25'),
(1722,1123,'market_25'),
(1722,1125,'market_25'),
(1722,1126,'market_25'),
(2203,700,'market_25'),
(2655,1124,'market_25'),
(3394,701,'market_25'),
(3396,700,'market_25'),
(10939,700,'market_50'),
(11678,703,'market_25'),
(12212,701,'market_50'),
(12212,703,'market_50'),
(12492,700,'market_25'),
(12492,701,'market_25'),
(12492,702,'market_25'),
(12492,703,'market_25'),
(12492,704,'market_25'),
(12492,1123,'market_25'),
(12492,1125,'market_25'),
(12492,1126,'market_25'),
(12493,700,'market_25'),
(12493,701,'market_25'),
(12493,703,'market_25'),
(12493,704,'market_25'),
(12493,1126,'market_25'),
(13453,1123,'market_max'),
(13453,1124,'market_50'),
(1223,704,'market_25'),
(9396,700,'market_25'),
(9396,701,'market_25'),
(9396,703,'market_25'),
(9396,704,'market_25'),
(13438,1123,'market_max'),
(19984,1126,'market_25'),
(4372,701,'market_25'),
(4372,1123,'market_25'),
(4372,1124,'market_25'),
(4372,1125,'market_25'),
(4372,1126,'market_25'),
(7523,701,'market_25'),
(14031,702,'market_25'),
(5642,702,'market_25'),
(5645,702,'market_25'),
(11059,702,'market_25'),
(12230,702,'market_max'),
(13484,701,'market_25'),
(21806,700,'market_25'),
(21806,701,'market_25'),
(21806,702,'market_25'),
(21806,703,'market_25'),
(21806,1126,'market_25'),
(21798,700,'market_25'),
(21798,701,'market_25'),
(21798,702,'market_50'),
(21809,700,'market_25'),
(21809,701,'market_25'),
(21809,702,'market_25'),
(21809,703,'market_25'),
(21809,1125,'market_25'),
(21809,1126,'market_25'),
(21810,700,'market_25'),
(21810,701,'market_25'),
(21810,702,'market_25'),
(21810,703,'market_25'),
(21810,1126,'market_25'),
(22438,701,'market_25'),
(22667,701,'market_25'),
(22667,1125,'market_25'),
(22706,1126,'market_25'),
(22865,700,'market_25'),
(22865,701,'market_25'),
(22865,1123,'market_25'),
(22865,1125,'market_25'),
(22865,1126,'market_25'),
(23028,702,'market_25'),
(23028,1123,'market_25'),
(23028,1124,'market_25'),
(23030,1126,'market_25'),
(23031,702,'market_25'),
(23031,1124,'market_25'),
(23031,1125,'market_25'),
(23603,700,'market_50'),
(23603,701,'market_50'),
(23603,1123,'market_50'),
(23603,1124,'market_50'),
(21807,700,'market_25'),
(21807,701,'market_25'),
(21807,702,'market_25'),
(21807,703,'market_25'),
(21807,1123,'market_25'),
(21807,1126,'market_25'),
(23334,700,'market_25'),
(23334,701,'market_25'),
(23334,702,'market_25'),
(19982,703,'market_75'),
(23601,1123,'market_25'),
(23601,1125,'market_25'),
(23602,1123,'market_25'),
(23598,1123,'market_25'),
(24228,701,'market_25'),
(24228,1125,'market_25'),
(24567,701,'market_25'),
(24668,703,'market_25'),
(20984,701,'market_25'),
(20984,1123,'market_25'),
(20984,1124,'market_25'),
(20984,1125,'market_25'),
(22361,1123,'market_25'),
(22361,1125,'market_25'),
(24705,701,'market_50'),
(24245,701,'market_25'),
(23382,700,'market_max'),
(23336,702,'market_25'),
(24125,702,'market_50'),
(24124,702,'market_50'),
(25378,701,'market_50'),
(25713,700,'market_25'),
(25713,701,'market_25'),
(25713,703,'market_25'),
(25713,704,'market_25'),
(25713,1123,'market_25'),
(25713,1125,'market_25'),
(25712,700,'market_25'),
(25712,701,'market_25'),
(25712,703,'market_25'),
(25712,704,'market_25'),
(25712,1123,'market_25'),
(25711,700,'market_25'),
(25711,701,'market_25'),
(25711,703,'market_25'),
(25711,704,'market_25'),
(25711,1123,'market_25'),
(25711,1125,'market_25'),
(25709,700,'market_25'),
(25709,701,'market_25'),
(25709,703,'market_25'),
(25709,704,'market_25'),
(25709,1125,'market_25'),
(25710,700,'market_25'),
(25710,701,'market_25'),
(25710,703,'market_25'),
(25710,704,'market_25'),
(25710,1125,'market_25'),
(22327,1126,'market_25'),
(13833,702,'market_50'),
(13833,703,'market_50'),
(13833,704,'market_25'),
(13833,1123,'market_50'),
(13833,1124,'market_50'),
(13833,1125,'market_50'),
(13833,1126,'market_50'),
(26685,700,'market_50'),
(26684,701,'market_50'),
(27012,702,'market_25'),
(25549,701,'market_75'),
(26593,700,'market_75'),
(26593,701,'market_75'),
(26593,702,'market_75'),
(26593,1126,'market_75'),
(25342,702,'market_25'),
(6206,700,'market_25'),
(6222,700,'market_25'),
(19931,700,'market_25'),
(19931,701,'market_25'),
(19931,702,'market_25'),
(19931,1123,'market_25'),
(19931,1126,'market_25') 
]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 370 OK across 160 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,2829,703,لمار عصير طبيعى تفاح- 200 مل,لمار عصاير,market_25,303.229780,370.50,332.2500,332.25,0.087345,OK
1,6802,704,توينكيز ميني اوريچينال - 24 عبوة,ايديتا,market_75,40.460000,45.00,43.8125,43.75,0.075200,OK
2,12870,700,كلوروكس ابيض باوش - 500 جم,كلوركس,market_25,190.820000,231.00,213.0000,213.00,0.104131,OK
3,12419,701,لمبادا ميني بسكويت.ويفر المحشو.بكريمة.الشيكولا...,لمبادا,market_75,91.136795,103.00,99.0000,99.00,0.079426,OK
4,5243,1125,جبنة ابو الولد - 40 مثلث,أبو الولد,market_25,116.146547,181.25,129.7500,129.75,0.104844,OK
...,...,...,...,...,...,...,...,...,...,...,...
365,19931,700,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,89.50,78.5000,78.50,0.095076,OK
366,19931,701,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,88.75,78.5000,78.50,0.095076,OK
367,19931,702,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,88.50,78.5000,78.50,0.095076,OK
368,19931,1123,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,89.50,78.5000,78.50,0.095076,OK


In [9]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.09464543238811694
0.09172844282771606
0.03664229854293724
0.17357026442473975


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,2829,703,لمار عصير طبيعى تفاح- 200 مل,لمار عصاير,market_25,303.229780,370.50,332.2500,332.25,0.087345,OK,-0.103239
1,6802,704,توينكيز ميني اوريچينال - 24 عبوة,ايديتا,market_75,40.460000,45.00,43.8125,43.75,0.075200,OK,-0.027778
2,12870,700,كلوروكس ابيض باوش - 500 جم,كلوركس,market_25,190.820000,231.00,213.0000,213.00,0.104131,OK,-0.077922
3,12419,701,لمبادا ميني بسكويت.ويفر المحشو.بكريمة.الشيكولا...,لمبادا,market_75,91.136795,103.00,99.0000,99.00,0.079426,OK,-0.038835
4,5243,1125,جبنة ابو الولد - 40 مثلث,أبو الولد,market_25,116.146547,181.25,129.7500,129.75,0.104844,OK,-0.284138
...,...,...,...,...,...,...,...,...,...,...,...,...
365,19931,700,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,89.50,78.5000,78.50,0.095076,OK,-0.122905
366,19931,701,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,88.75,78.5000,78.50,0.095076,OK,-0.115493
367,19931,702,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,88.50,78.5000,78.50,0.095076,OK,-0.112994
368,19931,1123,برسيل جل لافندر - 70 جم,برسيل,market_25,71.036536,89.50,78.5000,78.50,0.095076,OK,-0.122905


In [ ]:
# out = pd.read_excel('cohort-pricing-template.xlsx')
# id_ = 1255
# name = 'Mona_cairo'
# file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
# out.to_excel(file_name_,index = False,engine = 'xlsxwriter')
# time.sleep(3)
# ################### Loop to avoid file limit ######################
# # split file into chunks
# print('Spliting file into chunks...')
# if id_ == 61:
#     chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
# else:
#     chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
# print(f'len chunks = {len(chunks)}')
# fileslist = []
# for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
#     fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
#     output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
#     chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
# # Loop over chunks and upload
# print('Uploading...')
# for file in fileslist:
#     chunk = file.split('chunk_')[1].split('.xls')[0]
#     x = post_prices(id_, file)
#     # print(str(x.content))
#     if ('"success":true' in str(x.content).lower()):
#         print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
#     else:
#         print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
#         print(x.content)
#         final_status = False
#         break

In [10]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 160 products × 9 cohorts = 500 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 500
Price changes to push: 500
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 500

🔗 Mirrored prices to 6 main/general cohorts (+451 rows)
   Cohort 695 ← 700: 102 rows
   Cohort 61 ← 700: 102 rows
   Cohort 699 ← 702: 70 rows
   Cohort 697 ← 703: 56 rows
   Cohort 698 ← 704: 53 rows
   Cohort 696 ← 1123: 68 rows

📋 Prepared 1050 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (102 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 96.31it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (102 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 96.28it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (52 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 129.94it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 1050
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 500, 'price_changes': 500, 'pushed': 1050, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-19 11:50:22', 'mode': 'live', 'skipped_cohorts': []}
